# SCRIPT DESCRIPTION

Pseudobulk processing of full dataset (cohort dataset)

Filter full gene matrix (asap-cohort.merged_cleaned_unfiltered.h5ad) using the hvg adata and run pseudobulk analysis using Decoupler package. 

Filtering the large adata file (+7m cells) is computationally expensive. To reduce computation requirements, the full dataset is processed in chuncks (per sample) to filter cells and perform pseudobulk. After processing the cells for each sample the pseudobulk data is concatenated into one adata file.

Pseudobulk adata file is saved for downstream QC, preprocessing, exploratory analysis (e.g. gene expression across cell types and brain regions), and pathway analysis.


# 1. Package Imports and Configuration

In [ ]:
# Core scientific computing and visualization libraries
import numpy as np
import pandas as pd
import scanpy as sc
import decoupler as dc
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_auc_score, precision_recall_fscore_support, confusion_matrix

# Standard library imports
import sys
import subprocess
import importlib
import warnings
import os
from pathlib import Path
import gc

# Optional: enable cell-level timing for performance awareness
try:
    %load_ext autotime
except ModuleNotFoundError:
    %pip install ipython-autotime
    %load_ext autotime


# 1. Data structure

Define paths and input files

In [ ]:
#set general folder paths
HOME = Path.home()
WS_ROOT = HOME / "workspace"
DATA_DIR = WS_ROOT / "Data"
WS_FILES = WS_ROOT / "ws_files"
local_resources = WS_FILES / "local_resources"

if not WS_ROOT.exists():
    print(f"{WS_ROOT} doesn't exist. We need to remount our resources")
    !wb resource mount    

print("Home directory:     ", HOME)
print("Workspace root:     ", WS_ROOT)
print("Data directory:     ", DATA_DIR)
print("ws_files directory: ", WS_FILES)
print("Local CRN resources directory: ", local_resources)

print("\nContents of workspace root:")
for p in WS_ROOT.glob("*"):
    print(" -", p.name, "/" if p.is_dir() else "")

New folder structure for resulting files

In [ ]:
# set output folders

folder_name = "fulldataset"

# Define a local path for workshop files
data_path = WS_FILES / folder_name

# map my cells directories
input_dir = ( data_path / "input")
output_dir = ( data_path / "output")

# Make sure the directories exists
os.makedirs(data_path, exist_ok=True)
os.makedirs(input_dir, exist_ok=True)
os.makedirs(output_dir, exist_ok=True)

# Create the directory if it doesn't already exist
if not data_path.exists():
    print("Local folder missing")

print(f"Local data directory ready at: {data_path}")

CRN input data (e.g. HVG and full matrix AnnData) should be available in ~/workspace/ws_files/local_resources. If data does not exist, follow script 01.

adata ("asap-cohort.final.metadataset-integrated.h5ad") is the HVG files with dataset metadata included. It is ready to use for filtering cell types or brain regions of interest and subsetting the full dataset

In [ ]:
# set paths for raw data (hgv) files
adata_local_path = local_resources / f"asap-cohort.final.metadataset-integrated.h5ad"

# load adata and metadata files ("raw data")
adata_hvg = sc.read_h5ad(adata_local_path, backed="r")


In [ ]:
adata_hvg ##hvg adata

In [ ]:
# get cell order from hvg

clean_obs = adata_hvg.obs.copy()
valid_barcodes = clean_obs.index.values

In [ ]:
# set paths for raw data files
full_adata_path = local_resources / f"asap-cohort.merged_cleaned_unfiltered.h5ad"
adata_full = sc.read_h5ad(full_adata_path, backed="r")


In [ ]:
adata_full # full dataset

In [ ]:
# copy var from full adata
var_full = adata_full.var.copy()

### Filter full dataset based on "clean" hvg cells and perform pseudobulk (per sample)

Slice full data in chunks (unique sample) to avoid memory spikes

In [ ]:
# Get list of unique samples (~150-180 samples)
unique_samples = clean_obs["sample_clean"].unique()
pb_chunks = []

In [ ]:
print(f"Pseudobulking sample-by-sample across {len(unique_samples)} samples...")

for idx, sample_id in enumerate(unique_samples):
    # Find ALL barcodes belonging to THIS sample across the entire dataset
    sample_barcodes = clean_obs[clean_obs["sample_clean"] == sample_id].index

    # Load ONLY this sample's cells into memory
    sub_ad = adata_full[sample_barcodes, :].to_memory()
    sub_ad.obs = clean_obs.loc[sample_barcodes].copy()

    # Aggregate all cells for this complete sample
    sub_pb = dc.pp.pseudobulk(
        sub_ad,
        sample_col="sample_clean",
        groups_col="cell_type",
        mode="sum"
    )
    pb_chunks.append(sub_pb)

    if (idx + 1) % 10 == 0 or (idx + 1) == len(unique_samples):
        print(f"Processed {idx + 1}/{len(unique_samples)} samples...")

In [ ]:
# Concatenate tiny sample pseudobulks into final pdata (~200 MB)
print("Combining pseudobulks...")
pdata = sc.concat(pb_chunks)
pdata.var = var_full



In [ ]:
# Save output to disk
pseudobulk_save_path = input_dir / f"asap_full_genes_pseudobulk.h5ad"
pdata.write_h5ad(pseudobulk_save_path)
print("Saved pseudobulk adata saved successfully!")

In [ ]:
pdata